# Faster R-CNN / Mask R-CNN Training on Google Colab
Run this notebook on a GPU runtime (Runtime > Change runtime type > T4 GPU)

In [ ]:
# check gpu
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# clone the repo
!git clone https://github.com/martytcoleman/PA3.git
%cd PA3

In [ ]:
# install dependencies
!pip install torch torchvision tqdm pyyaml opencv-python matplotlib --quiet

In [ ]:
# download and extract VOC2007 dataset
import os

!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar
!tar -xf VOCtrainval_06-Nov-2007.tar
!tar -xf VOCtest_06-Nov-2007.tar

# create split directories
os.makedirs('VOC2007/JPEGImages', exist_ok=True)
os.makedirs('VOC2007/Annotations', exist_ok=True)
os.makedirs('VOC2007-test/JPEGImages', exist_ok=True)
os.makedirs('VOC2007-test/Annotations', exist_ok=True)

import subprocess

# symlink trainval split
with open('VOCdevkit/VOC2007/ImageSets/Main/trainval.txt') as f:
    for line in f:
        img_id = line.strip()
        src_img = os.path.abspath(f'VOCdevkit/VOC2007/JPEGImages/{img_id}.jpg')
        src_ann = os.path.abspath(f'VOCdevkit/VOC2007/Annotations/{img_id}.xml')
        dst_img = f'VOC2007/JPEGImages/{img_id}.jpg'
        dst_ann = f'VOC2007/Annotations/{img_id}.xml'
        if not os.path.exists(dst_img): os.symlink(src_img, dst_img)
        if not os.path.exists(dst_ann): os.symlink(src_ann, dst_ann)

# symlink test split
with open('VOCdevkit/VOC2007/ImageSets/Main/test.txt') as f:
    for line in f:
        img_id = line.strip()
        src_img = os.path.abspath(f'VOCdevkit/VOC2007/JPEGImages/{img_id}.jpg')
        src_ann = os.path.abspath(f'VOCdevkit/VOC2007/Annotations/{img_id}.xml')
        dst_img = f'VOC2007-test/JPEGImages/{img_id}.jpg'
        dst_ann = f'VOC2007-test/Annotations/{img_id}.xml'
        if not os.path.exists(dst_img): os.symlink(src_img, dst_img)
        if not os.path.exists(dst_ann): os.symlink(src_ann, dst_ann)

print('dataset ready')
print('trainval images:', len(os.listdir('VOC2007/JPEGImages')))
print('test images:', len(os.listdir('VOC2007-test/JPEGImages')))

In [ ]:
# quick sanity check
import sys
sys.path.insert(0, '.')
from train.test_implementation import (
    test_iou, test_sample_positive_negative,
    test_clamp_boxes, test_transform_boxes,
    test_rpn, test_roi_head, test_full_model
)
test_iou()
test_sample_positive_negative()
test_clamp_boxes()
test_transform_boxes()
test_rpn()
test_roi_head()
test_full_model()
print('all tests passed')

In [ ]:
# train faster r-cnn
!python train/train_faster_rcnn.py --config config/voc.yaml

In [ ]:
# evaluate mAP on test set
!python test/test_faster_rcnn.py --config config/voc.yaml --evaluate True --infer_samples True

In [ ]:
# train mask r-cnn (bonus)
!python train/train_mask_rcnn.py --config config/voc_mask.yaml

In [ ]:
# save checkpoints to google drive (optional)
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('voc/faster_rcnn_voc2007.pth', '/content/drive/MyDrive/PA3_faster_rcnn.pth')
print('checkpoint saved to drive')